# Week 2 Day 5 Challenge - the short policy wording as a lever

The challenge: give the improver another lever. Add the concise policy wording in `data/policy_wording_short.md` as a choice in the configuration, and see whether the improver finds a cheaper prompt that still meets the objectives.

The claim agent, the judge, the dispatcher, the scoring and the improver are the lab's, with these changes:

| | Lab | Challenge |
|---|---|---|
| The configuration | Instructions, model, reasoning level | The same, and `policy_wording`: `full` or `short` |
| The claim agent's system prompt | The instructions and the full wording | The instructions and the wording the configuration names |
| `propose_change` | Instructions, model, reasoning level | The same, and the wording |
| `read_results` | The current version, scores, mistakes, rules, budget, models | The same, and both wordings in full, so the improver can see what the short one leaves out |
| The recipe's step 3 | Keep the model and the reasoning level | The same, and the improver may switch to the short wording if it should cut the cost without losing accuracy or grade |
| The scores | Accuracy, grade, cost and time | The same, and Jev's same-section answer, the prompt tokens per claim and the share of them served from the prompt cache |
| A control | None | Version 1 with the short wording and nothing else changed, run by hand before the improver starts, so we know what the swap alone does |
| The budget | $0.50 | $1.00, since the control adds three runs |
| Rounds | Two, approved by hand | Three, approved by the notebook, with every proposal printed above its approval |

Two things make this lever less simple than it looks:

1. The short wording has no section numbers. Its sections are called Limits, Trip Delay, Payment and so on, while version 1's instructions say "Work out the payout using section 10" and the judge checks the section against the handler's reference.
2. OpenAI caches the start of a prompt it has seen recently. The claim agent's system prompt starts with the instructions, so any change to them makes the prompt new to the cache, whichever wording follows.

The models are the lab's: GPT-6 Luna on OpenRouter for the claim agent (at `low`) and the improver (at `medium`), and Jev as the judge.

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT / "labs"))

In [ ]:
import difflib
import json
import operator
import os
from datetime import datetime
from typing import Annotated, Literal, TypedDict

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langfuse import get_client
from langfuse.langchain import CallbackHandler
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, Send
from openrouter import OpenRouter
from week2day1_payout import Benefit, Line, Payout, work_out_payout

load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
print("OpenRouter API key found" if openrouter_api_key else "No OpenRouter API key found")

langfuse = get_client()
langfuse_handler = CallbackHandler()
print("Connected to Langfuse" if langfuse.auth_check() else "No Langfuse keys found")

## The data and the two wordings

The lab's loader, with the evaluation set and the held-out set. The improver only ever reads the evaluation set's claims.

In [ ]:
DATA = ROOT / "data"


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    return claims, {decision["claim_id"]: decision for decision in decisions}


evaluation_claims, evaluation_labels = get_sample_data()
held_out_claims, held_out_labels = get_sample_data(held_out=True)

claims = {claim["claim_id"]: claim for claim in evaluation_claims + held_out_claims}
labels = evaluation_labels | held_out_labels
policies = {policy["policy_id"]: policy for policy in load("policies.json")}

SETS = {"evaluation": list(evaluation_labels), "held-out": list(held_out_labels)}
print(f"{len(SETS['evaluation'])} evaluation claims and {len(SETS['held-out'])} held-out claims")

In [ ]:
WORDINGS = {
    "full": (DATA / "policy_wording.md").read_text(encoding="utf-8"),
    "short": (DATA / "policy_wording_short.md").read_text(encoding="utf-8"),
}

for name, text in WORDINGS.items():
    headings = [line.lstrip("# ") for line in text.splitlines() if line.startswith("## ")]
    print(f"{name}: {len(text):,} characters. Sections: {', '.join(headings)}")

## The judge

Jev with the lab's rubric. The lab calibrated it on CLM-1001, so it isn't calibrated again here.

In [ ]:
JEV = "typesafe/jev-1.13"

RUBRIC = {
    "F": "different grounds from the handler's reason, or it contradicts the handler's reason",
    "C": "partly the same as the handler's reason: one of the handler's grounds or the section is missing, or it adds a ground the handler did not give",
    "B": "the same grounds and the same section of the policy wording as the handler's reason",
    "A": "the same grounds and section as the handler's reason, and clearly better expressed: clearer for the customer, with the section and the figures, shorter, and no jargon",
}
GRADES = list(RUBRIC)

JUDGE_QUESTIONS = {
    "grade": {
        "type": "score",
        "instructions": "An expert claims handler decided a travel insurance claim and gave `handler.reason`, resting on `handler.reference`. Grade how well `agent.rationale` explains the same decision.",
        "criteria": [f"{letter}: {text}" for letter, text in RUBRIC.items()],
    },
    "same_grounds": {"type": "noul", "instructions": "`agent.rationale` gives the same main grounds for the decision as `handler.reason`"},
    "same_section": {"type": "noul", "instructions": "`agent.rationale` refers to the same section of the policy wording as `handler.reference`"},
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def judge(rationale, decision, expected):
    """Ask Jev to grade a rationale against the handler's reason."""
    state = {
        "handler": {"decision": expected["decision"], "reason": expected["reason"], "reference": expected["reference"]},
        "agent": {"decision": decision, "rationale": rationale},
    }
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions=JUDGE_QUESTIONS)
    answers = response.answers
    probabilities = answers["grade"].probabilities
    most_likely = max(probabilities, key=probabilities.get)
    return {
        "grade": answers["grade"].score,
        "letter": GRADES[int(most_likely)],
        "same_grounds": answers["same_grounds"].noul,
        "same_section": answers["same_section"].noul,
        "judge_cost": response.usage.cost,
    }

## The configuration and the claim agent

Version 1 is the lab's, with the full wording. `make_agent` puts the wording the configuration names after the instructions.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
Use the payout tool to work out the payout on an approved claim, and record the payout it returns.
"""

versions = [{"version": 1, "instructions": INSTRUCTIONS, "model": "openai/gpt-6-luna", "reasoning_effort": "low", "policy_wording": "full"}]

# The current version and the money spent, in a dict so that the tools can change them without `global`
progress = {"current": 1, "spent": 0.0}

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_agent(config):
    """Build the claim agent for one version of the configuration, with the wording it names."""
    model = ChatOpenAI(model=config["model"], base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                       reasoning_effort=config["reasoning_effort"], timeout=120, max_retries=5)
    return create_agent(model=model, tools=[record_decision, calculate_payout],
                        system_prompt=f"{config['instructions']}\n{WORDINGS[config['policy_wording']]}\n")

## The dispatcher and the scores

The lab's dispatcher. `work_claim` also adds up the prompt tokens of the claim agent's model calls and how many of them came from the prompt cache.

In [ ]:
class Dispatch(TypedDict):
    """What the dispatcher keeps for one run."""

    config: dict
    claims: list[str]
    results: Annotated[list, operator.add]


def fan_out(state: Dispatch):
    """Send each claim to a claim agent, with the configuration to use."""
    return [Send("work_claim", {"claim_id": claim_id, "config": state["config"]}) for claim_id in state["claims"]]


def work_claim(task):
    """Run a claim agent on one claim and return its result, with its cost and prompt tokens."""
    claim_id = task["claim_id"]
    agent = make_agent(task["config"])
    started = datetime.now()
    result = agent.invoke({"messages": [{"role": "user", "content": make_user_prompt(claims[claim_id])}]})
    answer = recorded.get(claim_id, {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    usage = [message.response_metadata["token_usage"] for message in result["messages"] if message.type == "ai"]
    return {"results": [{
        "claim_id": claim_id, **answer,
        "cost": sum(u["cost"] for u in usage),
        "prompt_tokens": sum(u["prompt_tokens"] for u in usage),
        "cached_tokens": sum((u.get("prompt_tokens_details") or {}).get("cached_tokens") or 0 for u in usage),
        "started": started, "finished": datetime.now(),
    }]}


builder = StateGraph(Dispatch)
builder.add_node("work_claim", work_claim, retry_policy=RetryPolicy(max_attempts=3))
builder.add_conditional_edges(START, fan_out, ["work_claim"])
builder.add_edge("work_claim", END)
dispatcher = builder.compile()

In [ ]:
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def score_table(results):
    """One row for each claim: the agent's answer beside the handler's, with the judge's grade, cost, tokens and time."""
    rows = []
    for answer in results:
        expected = labels[answer["claim_id"]]
        rows.append({
            "claim_id": answer["claim_id"],
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            **judge(answer["rationale"], answer["decision"], expected),
            "llm_cost": answer["cost"],
            "prompt_tokens": answer["prompt_tokens"],
            "cached_tokens": answer["cached_tokens"],
            "seconds": (answer["finished"] - answer["started"]).total_seconds(),
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run: quality, cost, prompt tokens and time."""
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "grade": table["grade"].mean(),
        "same_section": table["same_section"].mean(),
        "cost_per_claim": table["llm_cost"].mean(),
        "prompt_tokens_per_claim": table["prompt_tokens"].mean(),
        "cached_share": table["cached_tokens"].sum() / table["prompt_tokens"].sum(),
        "seconds_per_claim": table["seconds"].mean(),
    }

`evaluate` is the lab's, with the wording on the scoreboard.

In [ ]:
tables = {}
scoreboard = []
run_costs = []  # each run's cost per claim, the claim agent and the judge together
session = f"Challenge {datetime.now():%Y-%m-%d %H:%M}"


def evaluate(config, set_name):
    """Run one version of the configuration on one set, score it and add it to the scoreboard."""
    recorded.clear()
    run_name = f"Version {config['version']} on the {set_name} set"
    result = dispatcher.invoke({"config": config, "claims": SETS[set_name]},
                               {"max_concurrency": 25, "callbacks": [langfuse_handler], "run_name": run_name,
                                "metadata": {"langfuse_session_id": session}})
    table = score_table(result["results"])
    scores = run_scores(table)
    progress["spent"] += table["llm_cost"].sum() + table["judge_cost"].sum()
    run_costs.append((table["llm_cost"] + table["judge_cost"]).mean())
    tables[(config["version"], set_name)] = table
    scoreboard.append({"version": config["version"], "wording": config["policy_wording"], "set": set_name, **scores})
    print(f"{run_name} ({config['policy_wording']} wording): accuracy {scores['accuracy']:.2f}, grade {scores['grade']:.2f}, "
          f"${scores['cost_per_claim']:.5f}, {scores['prompt_tokens_per_claim']:,.0f} prompt tokens "
          f"({scores['cached_share']:.0%} cached) and {scores['seconds_per_claim']:.1f} seconds a claim")
    return scores


def show_scoreboard():
    """The scoreboard, with the columns that matter for this challenge."""
    columns = ["version", "wording", "set", "accuracy", "grade", "same_section", "cost_per_claim",
               "prompt_tokens_per_claim", "cached_share", "seconds_per_claim"]
    return pd.DataFrame(scoreboard)[columns].round(4)

## Version 1

As in the lab: once on the evaluation set and twice on the held-out set, so we know how much the scores move with no change.

In [ ]:
evaluate(versions[0], "evaluation")
evaluate(versions[0], "held-out")
evaluate(versions[0], "held-out")

show_scoreboard()

## The rules for keeping a version

The lab's objectives, noise, tolerance and models allowed, and its keep-or-roll-back rule. The budget is $1.00.

In [ ]:
OBJECTIVES = {"accuracy": 0.6, "grade": 1.0}
NOISE = {"accuracy": 0.1, "grade": 0.15, "cost_per_claim": 0.15, "seconds_per_claim": 0.15}
TOLERANCE = {"accuracy": 0.1, "grade": 0.15, "cost_per_claim": 1.0, "seconds_per_claim": 0.25}
BUDGET = 1.00

ALLOWED_MODELS = {
    "openai/gpt-6-luna": {"reasoning_efforts": ["none", "low", "medium"], "price per million tokens": "$0.10 in, $0.50 out"},
    "google/gemini-3.5-flash-lite": {"reasoning_efforts": ["minimal", "low"], "price per million tokens": "$0.30 in, $2.50 out"},
    "deepseek/deepseek-v4.1-flash": {"reasoning_efforts": ["low"], "price per million tokens": "$0.30 in, $1.20 out"},
}


def compare(new, old):
    """Which scores got better by more than the noise, and which got worse by more than the tolerance."""
    better, worse = [], []
    for name in ["accuracy", "grade"]:
        if new[name] > old[name] + NOISE[name]:
            better.append(name)
        if new[name] < old[name] - TOLERANCE[name]:
            worse.append(name)
    for name in ["cost_per_claim", "seconds_per_claim"]:
        if new[name] < old[name] * (1 - NOISE[name]):
            better.append(name)
        if new[name] > old[name] * (1 + TOLERANCE[name]):
            worse.append(name)
    return better, worse


def keep_or_roll_back(new, old):
    """Decide whether to keep a new version from its held-out scores and the current version's average."""
    missed = [name for name, floor in OBJECTIVES.items() if new[name] < floor]
    better, worse = compare(new, old)
    if missed:
        return False, f"it missed the objectives on {', '.join(missed)}"
    if worse:
        return False, f"{', '.join(worse)} got worse"
    if not better:
        return False, "no score improved by more than the noise"
    return True, f"{', '.join(better)} improved"


def average(version, set_name):
    """The average of a version's scores over all its runs on a set."""
    runs = pd.DataFrame([row for row in scoreboard if row["version"] == version and row["set"] == set_name])
    return runs.drop(columns=["version", "wording", "set"]).mean().to_dict()


def latest(version, set_name):
    """The most recent scores for a version on a set."""
    return [row for row in scoreboard if row["version"] == version and row["set"] == set_name][-1]

## The control: the short wording and nothing else

Version 1's instructions with the short wording, run like a new version: once on the evaluation set and twice on the held-out set. It isn't a version the improver can see or build on. It tells us what the swap alone does, and what the keep-or-roll-back rule would make of it.

In [ ]:
control = {**versions[0], "version": "1-short", "policy_wording": "short"}

evaluate(control, "evaluation")
evaluate(control, "held-out")
evaluate(control, "held-out")

keep, why = keep_or_roll_back(average("1-short", "held-out"), average(1, "held-out"))
print(f"\nThe rule would {'keep' if keep else 'roll back'} the swap alone: {why}")
show_scoreboard()

## The improver

The lab's tools, with the wording added to `read_results` and `propose_change`. The control stays out of what the improver reads: `read_results` reports only the current version and the earlier rounds.

In [ ]:
rounds = []
recipes = []


def beside_handler(table):
    """Each claim in a table with the agent's answer beside the handler's and the judge's grade."""
    rows = []
    for row in table.to_dict("records"):
        expected = labels[row["claim_id"]]
        rows.append({
            "claim_id": row["claim_id"],
            "summary": claims[row["claim_id"]]["summary"],
            "agent": {"decision": row["agent"], "payout": row["agent_payout"], "rationale": row["rationale"]},
            "handler": {"decision": expected["decision"], "payout": expected["payout"], "reason": expected["reason"], "reference": expected["reference"]},
            "judge": {"grade": round(row["grade"], 2), "letter": row["letter"]},
        })
    return rows


def mistakes(version):
    """The evaluation claims a version got wrong, and the five right ones with the lowest grades."""
    table = tables[(version, "evaluation")]
    weakest = table[table["right"]].sort_values("grade").head(5)
    return {"wrong": beside_handler(table[~table["right"]]), "explained_least_well": beside_handler(weakest)}


@tool
def read_results() -> str:
    """Read the current version of the claim agent's configuration, its scores, its mistakes and weakest explanations on the evaluation set, the earlier rounds, the objectives, the budget, the models allowed and the two policy wordings."""
    current = progress["current"]
    config = versions[current - 1]
    report = {
        "current_version": config,
        "scores": {set_name: average(current, set_name) for set_name in SETS},
        "evaluation_set": mistakes(current),
        "earlier_rounds": rounds,
        "objectives_on_the_held_out_set": OBJECTIVES,
        "noise": NOISE,
        "tolerance": TOLERANCE,
        "rule": "A new version is kept when it meets the objectives, at least one score improves by more than the noise and none gets worse by more than the tolerance",
        "budget_left": round(BUDGET - progress["spent"], 4),
        "models_allowed": ALLOWED_MODELS,
        "policy_wordings": WORDINGS,
    }
    return json.dumps(report, indent=2, default=str)


@tool(parse_docstring=True)
def propose_change(instructions: str, model: str, reasoning_effort: str, policy_wording: Literal["full", "short"], reason: str) -> str:
    """Propose a new version of the claim agent's configuration. A human reviews it before it's saved.

    Args:
        instructions: The claim agent's complete instructions
        model: One of the models allowed
        reasoning_effort: A reasoning level allowed for that model
        policy_wording: The policy wording that follows the instructions in the claim agent's prompt, full or short
        reason: Why this change should improve the scores
    """
    if reasoning_effort not in ALLOWED_MODELS.get(model, {}).get("reasoning_efforts", []):
        return f"{model} at {reasoning_effort} isn't allowed. Choose from the models allowed."
    version = len(versions) + 1
    versions.append({"version": version, "instructions": instructions, "model": model, "reasoning_effort": reasoning_effort,
                     "policy_wording": policy_wording, "reason": reason})
    return f"Saved as version {version}. Run the evaluation next."


@tool
def run_evaluation() -> str:
    """Run the newest version once on the evaluation set and twice on the held-out set, then keep it or roll it back."""
    config = versions[-1]
    current = progress["current"]
    if config["version"] == current or any(r["version"] == config["version"] for r in rounds):
        return "There's no new version to evaluate. Propose a change first."
    estimate = (len(SETS["evaluation"]) + 2 * len(SETS["held-out"])) * max(run_costs)
    if progress["spent"] + estimate > BUDGET:
        return f"Refused: the evaluation could cost about ${estimate:.2f} and ${BUDGET - progress['spent']:.2f} of the budget is left"
    before = average(current, "held-out")
    evaluate(config, "evaluation")
    evaluate(config, "held-out")
    evaluate(config, "held-out")
    after = average(config["version"], "held-out")
    keep, why = keep_or_roll_back(after, before)
    rounds.append({"round": len(rounds) + 1, "recipe_version": len(recipes), "version": config["version"],
                   "model": config["model"], "reasoning_effort": config["reasoning_effort"], "policy_wording": config["policy_wording"],
                   "reason": config["reason"], "kept": keep, "why": why, "held_out_before": before, "held_out_after": after})
    if keep:
        current = progress["current"] = config["version"]
    verdict = "Kept" if keep else "Rolled back"
    scores = {"held_out_before": before, "held_out_after": after, "evaluation_after": latest(config["version"], "evaluation")}
    return f"{verdict} version {config['version']}: {why}. Version {current} is now the current version.\n{json.dumps(scores, indent=2)}"


@tool(parse_docstring=True)
def propose_recipe(recipe: str, reason: str) -> str:
    """Propose a new version of your recipe for improving the claim agent. A human reviews it before it's saved.

    Args:
        recipe: The complete new recipe
        reason: What this round showed, and why the new recipe should lead to better changes
    """
    recipes.append(recipe)
    return f"Saved as recipe version {len(recipes)}."

### The recipe

The lab's recipe, with the wording added to step 3.

In [ ]:
RECIPE = """1. Call read_results to see how the current version is doing and which evaluation claims it got wrong or explained poorly.
2. For each of those claims, add a rule to the instructions that would make the agent get that claim right.
3. Call propose_change with the new instructions, keeping the same model and reasoning level. You may switch the policy wording to the short version if that should cut the cost per claim without losing accuracy or grade.
4. Call run_evaluation to test the new version on both sets.
5. Improve this recipe, so that the next round makes a better change. Look at what happened in this round and in earlier rounds, then call propose_recipe with the new recipe. Keep this as the last step of the new recipe.
"""

recipes.append(RECIPE)

In [ ]:
improver_model = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key,
                            reasoning_effort="medium", timeout=300, max_retries=5)

approval = HumanInTheLoopMiddleware(interrupt_on={
    "propose_change": {"allowed_decisions": ["approve", "reject"]},
    "propose_recipe": {"allowed_decisions": ["approve", "reject"]},
})


def make_improver(recipe):
    """Build the improver with a recipe as its instructions."""
    system_prompt = f"""You improve the configuration of UnderCover's claim agent, an AI agent that decides travel insurance claims.
The claim agent sees its instructions, then the policy wording its configuration names, then the customer's policy record and the claim. It doesn't see the claims manual or the customer's history.
Follow your recipe one step at a time.

# Your recipe

{recipe}
"""
    return create_agent(model=improver_model, tools=[read_results, propose_change, run_evaluation, propose_recipe],
                        system_prompt=system_prompt, middleware=[approval], checkpointer=InMemorySaver())

`show_proposal` is the lab's, and also prints a change of wording. `run_round` carries out one round of the recipe, printing each proposal before it approves it. A notebook that runs from top to bottom can't wait for a person, so the approvals are in the code; the proposals printed are what I would have read.

In [ ]:
APPROVE = Command(resume={"decisions": [{"type": "approve"}]})


def show_changed_lines(old, new):
    """Print the lines that differ between two texts."""
    for line in difflib.unified_diff(old.splitlines(), new.splitlines(), lineterm="", n=0):
        if not line.startswith(("---", "+++", "@@")):
            print(line)


def show_proposal(result):
    """Show the change the improver is waiting for us to approve."""
    request = result["__interrupt__"][0].value["action_requests"][0]
    args = request["args"]
    if request["name"] == "propose_change":
        old = versions[progress["current"] - 1]
        print(f"Proposed change. Model: {old['model']} at {old['reasoning_effort']}, proposed {args['model']} at {args['reasoning_effort']}")
        print(f"Wording: {old['policy_wording']}, proposed {args['policy_wording']}\n")
        show_changed_lines(old["instructions"], args["instructions"])
    else:
        print(f"Proposed recipe. Recipe version {len(recipes)}, proposed version {len(recipes) + 1}\n")
        show_changed_lines(recipes[-1], args["recipe"])
    print(f"\nReason: {args['reason']}\n")


def run_round(number):
    """Carry out one round of the latest recipe, approving each proposal after printing it."""
    improver = make_improver(recipes[-1])
    config = {"configurable": {"thread_id": f"challenge-round-{number}"}, "callbacks": [langfuse_handler],
              "run_name": f"Improver round {number}", "metadata": {"langfuse_session_id": session}}
    result = improver.invoke({"messages": [{"role": "user", "content": "Please carry out one round of your recipe."}]}, config)
    for _ in range(5):
        if "__interrupt__" not in result:
            break
        show_proposal(result)
        result = improver.invoke(APPROVE, config)
    print(f"The improver's last message:\n{result['messages'][-1].content}")


def show_rounds():
    """One row for each round: the version it tried, its wording, the held-out scores before and after, and the verdict."""
    rows = []
    for r in rounds:
        rows.append({
            "round": r["round"], "recipe": r["recipe_version"], "version": r["version"],
            "model": r["model"], "reasoning": r["reasoning_effort"], "wording": r["policy_wording"],
            "accuracy_before": r["held_out_before"]["accuracy"], "accuracy_after": r["held_out_after"]["accuracy"],
            "grade_before": r["held_out_before"]["grade"], "grade_after": r["held_out_after"]["grade"],
            "cost_before": r["held_out_before"]["cost_per_claim"], "cost_after": r["held_out_after"]["cost_per_claim"],
            "kept": r["kept"], "why": r["why"],
        })
    return pd.DataFrame(rows).round(5)

## Round 1

In [ ]:
run_round(1)
show_rounds()

## Round 2

In [ ]:
run_round(2)
show_rounds()

## Round 3

In [ ]:
run_round(3)
show_rounds()

## The results

Every run of every version, with the control as `1-short`.

In [ ]:
show_scoreboard()

The average of each version's runs on the held-out set, beside the objectives.

In [ ]:
held_out = pd.DataFrame(scoreboard).query("set == 'held-out'")
summary = held_out.groupby(["version", "wording"], sort=False)[
    ["accuracy", "grade", "same_section", "cost_per_claim", "prompt_tokens_per_claim", "cached_share", "seconds_per_claim"]
].mean()
summary["meets_objectives"] = (summary["accuracy"] >= OBJECTIVES["accuracy"]) & (summary["grade"] >= OBJECTIVES["grade"])
summary.round(4)

In [ ]:
final = versions[progress["current"] - 1]
print(f"Current version: {final['version']}, {final['model']} at {final['reasoning_effort']} with the {final['policy_wording']} wording")
print(f"Spent: ${progress['spent']:.4f} of ${BUDGET:.2f}\n")
print("Its instructions, against version 1's:")
show_changed_lines(versions[0]["instructions"], final["instructions"])

In [ ]:
print(recipes[-1])

## What I found

**The improver didn't take the short wording, and it was right not to.** It could read both wordings in full, and in every round it kept the full one. In round 1 it "retained the full wording and model/settings to isolate the instruction change", and the recipe it wrote after round 1 made the full wording the default, with the short one allowed only "if it preserves all decision-relevant rules and is likely to lower cost". Among the configurations tried, there was no cheaper prompt that kept the grade and saved more than the cost noise:

| On the held-out set, two runs each | Version 1, full wording | Version 1, short wording |
|---|---|---|
| Accuracy | 0.70 | 0.60 |
| Grade | 1.24 | 0.81 |
| Same section as the handler (Jev's probability) | 0.75 | 0.49 |
| Prompt tokens a claim | 9,082 | 7,132 |
| Share of them from the cache | 98% | 92% |
| Cost a claim | $0.00026 | $0.00028 |
| Seconds a claim | 5.7 | 6.8 |

The keep-or-roll-back rule would have rolled the swap back: the grade fell below the objective of 1.0. Accuracy fell by 0.10, exactly the tolerance, so on its own it wouldn't have counted as worse; the grade decided it.

**The grade fell mostly because the short wording has no section numbers.** The handlers' references say "Policy wording, section 3", and the rubric's B needs the same section. With the short wording the agent can only name a section, such as "Trip Delay", and Jev's same-section answer fell from 0.75 to 0.49. Version 1's instructions still say "Work out the payout using section 10", which the short wording doesn't have. A short wording with the same numbers would be a fairer test of this lever.

**The cost didn't fall, and it couldn't have fallen by much.** The short wording cut the prompt by 1,950 tokens a claim, 21%, but the prompt was already almost free. Luna charges $0.10 per million input tokens and, by OpenRouter's price list, $0.01 per million cached ones, and 98% of the full prompt came from the cache, so the full prompt cost about $0.00011 a claim. The short prompt was new to the cache: 84%, 86% and then 97% of it was cached over its three runs, so on average it cost more, about $0.00013. Even once the cache is warm, the saving is about $0.00002 a claim, 7% of the cost. The cost noise is 15%, so a cheaper wording on its own could never clear it. Most of the cost is the agent's output.

**The rounds themselves showed the overfitting the lab warns about.** All three rounds were rolled back:

| Round | Change | Evaluation accuracy | Held-out accuracy | Held-out grade | Verdict |
|---|---|---|---|---|---|
| | Version 1 | 0.68 | 0.70 | 1.24 | |
| 1 | Six rules from the wrong claims | 0.76 | 0.74 | 1.25 | No score past the noise |
| 2 | Broader rules on exclusions and referrals | 0.68 | 0.58 | 1.03 | Below the accuracy objective |
| 3 | One narrow rule on unattended baggage | 0.68 | 0.72 | 1.22 | No score past the noise |

Round 1's rules included one written for a single claim: "For the sleeper-train delay described as 30 hours, ... one traveler receives $70 for meals". A person reading the proposal should have rejected it. This notebook approves every proposal so it can run from top to bottom, so it went through. The evaluation set rewarded the round (0.68 to 0.76); on the held-out set it didn't get past the noise. Round 2 left the evaluation accuracy where it was and cost 0.12 on the held-out set. In the lab's run, a similar first round was kept, but for its grade (1.19 to 1.38, past the 0.15 noise). Its accuracy (0.68 to 0.78) didn't clear the noise either. Here the grade didn't move (1.242 to 1.245), so the round was rolled back.

**The recipe improved in the direction the results pointed.** Each new recipe asked for fewer and narrower rules, a check for the cases a rule could harm, and "no change" as an option. Round 3 followed it with a single rule. The improver's summaries weren't always accurate, though: after round 1 it reported the evaluation accuracy as 0.68 to 0.84 and the held-out as 0.72 to 0.78, which are not the scoreboard's accuracy (0.76 and 0.74) and are most likely its decision accuracy, which ignores the payout. Recipe 2 copied the 0.84 into itself, so later rounds read it. The keep-or-roll-back verdicts are worked out in code, so they don't depend on these summaries. Each recipe also copied the latest round's numbers into itself, and recipe 4 is mostly a report of round 3. A recipe that grows with every round would need its history moved back out to `read_results`.

**For the product.** The wording is a lever worth keeping in the configuration, but only with the same section numbers as the full wording, and with cost judged once the prompt cache is warm. A cheaper prompt does more for cost when it comes after the part of the prompt that stays the same. The claim agent's system prompt starts with the instructions, so every instruction change makes the whole prompt new to the cache.

**What I didn't check.** The lab's fallbacks are kept as they are: a claim agent that records nothing counts as a referral, and a retried claim keeps only the last attempt's cost. The notebook doesn't count how often either happened.

**Cost.** $0.13 on evaluations of the $1.00 budget. With the improver's own calls it came to about $0.15, from the OpenRouter key's usage for the day. Models: GPT-6 Luna and Jev through OpenRouter.